# DiGiTerra Classification Workflow Notebook

This notebook mirrors Model Preprocessing + Modeling for classification, with class labels built from binned target values.

Target bins used by default:
- 0-49
- 50-149
- 150-249
- 250-499
- 500-999
- 1000+


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAS_SEABORN = True
except Exception:
    _HAS_SEABORN = False

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 1) Configuration


In [2]:
CONFIG = {
    'input_path': Path('/Users/rowanterra/Desktop/Test_1217/mapping/All_Sites_Test_3_1.xlsx'),

    # Training source
    'train_sheet': 'Literature',
    'train_target': 'TREE',

    # External source (REEY considered equivalent if desired)
    'external_sheet': 'Backvalidation',
    'external_target': 'REEY',

    'features': ['pH', 'Fe', 'Mn', 'Al', 'SO4'],
    'test_size': 0.2,
    'single_seed': 42,
    'cv_folds': 5,

    'seed_start': 0,
    'seed_count': 30,

    # Binning for classification labels
    'bin_edges': [-0.000001, 49, 149, 249, 499, 999, float('inf')],
    'bin_labels': ['0-49', '50-149', '150-249', '250-499', '500-999', '1000+'],

    'drop_missing': 'indicatorAndTarget',

    'candidates': [
        {'name': 'ExtraTrees', 'outlier_mode': 'winsorize_1_99'},
        {'name': 'RandomForest', 'outlier_mode': 'winsorize_1_99'},
        {'name': 'KNN', 'outlier_mode': 'none'},
        {'name': 'MLP', 'outlier_mode': 'winsorize_1_99'},
    ],

    'use_grid_search': True,
    'grid_n_jobs': -1,
}

PARAM_GRIDS = {
    'ExtraTrees': {
        'model__n_estimators': [300, 500],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_leaf': [1, 2, 4],
    },
    'RandomForest': {
        'model__n_estimators': [300, 500],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_leaf': [1, 2, 4],
    },
    'KNN': {
        'model__n_neighbors': [5, 7, 9, 11],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2],
    },
    'MLP': {
        'model__hidden_layer_sizes': [(64,), (128, 64), (256, 128)],
        'model__alpha': [1e-5, 1e-4, 1e-3],
        'model__learning_rate_init': [1e-3, 5e-4],
        'model__activation': ['relu', 'tanh'],
    },
    'Logistic': {
        'model__C': [0.1, 1.0, 10.0],
    },
}

CONFIG


{'input_path': PosixPath('/Users/rowanterra/Desktop/Test_1217/mapping/All_Sites_Test_3_1.xlsx'),
 'train_sheet': 'Literature',
 'train_target': 'TREE',
 'external_sheet': 'Backvalidation',
 'external_target': 'REEY',
 'features': ['pH', 'Fe', 'Mn', 'Al', 'SO4'],
 'test_size': 0.2,
 'single_seed': 42,
 'cv_folds': 5,
 'seed_start': 0,
 'seed_count': 30,
 'bin_edges': [-1e-06, 49, 149, 249, 499, 999, inf],
 'bin_labels': ['0-49', '50-149', '150-249', '250-499', '500-999', '1000+'],
 'drop_missing': 'indicatorAndTarget',
 'candidates': [{'name': 'ExtraTrees', 'outlier_mode': 'winsorize_1_99'},
  {'name': 'RandomForest', 'outlier_mode': 'winsorize_1_99'},
  {'name': 'KNN', 'outlier_mode': 'none'},
  {'name': 'MLP', 'outlier_mode': 'winsorize_1_99'}],
 'use_grid_search': True,
 'grid_n_jobs': -1}

## 2) Utility Functions


In [3]:
class Winsorizer(TransformerMixin, BaseEstimator):
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(arr, self.lower_q, axis=0)
        self.upper_ = np.nanquantile(arr, self.upper_q, axis=0)
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        return np.clip(arr, self.lower_, self.upper_)


def load_tabular(path: Path, sheet_name: str | None = None) -> pd.DataFrame:
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        if sheet_name is None:
            raise ValueError('sheet_name is required for Excel files')
        return pd.read_excel(path, sheet_name=sheet_name)
    return pd.read_csv(path)


def clean_dataframe(df: pd.DataFrame, features: list[str], target: str, drop_missing='indicatorAndTarget'):
    cols = features + [target]
    out = df.copy()
    out[cols] = out[cols].apply(pd.to_numeric, errors='coerce')
    if drop_missing == 'indicatorAndTarget':
        out = out.dropna(subset=cols, how='any')
    elif drop_missing == 'indicator':
        out = out.dropna(subset=features, how='any')
    elif drop_missing == 'target':
        out = out.dropna(subset=[target], how='any')
    elif drop_missing == 'all':
        out = out.dropna(how='all')
    return out.reset_index(drop=True)


def bin_target(y: pd.Series, edges: list[float], labels: list[str]) -> pd.Series:
    y_num = pd.to_numeric(y, errors='coerce')
    b = pd.cut(y_num, bins=edges, labels=labels, include_lowest=True, right=True)
    return b.astype(str)


def get_model(model_name: str, seed: int):
    if model_name == 'ExtraTrees':
        return ExtraTreesClassifier(random_state=seed, n_jobs=-1)
    if model_name == 'RandomForest':
        return RandomForestClassifier(random_state=seed, n_jobs=-1)
    if model_name == 'KNN':
        return KNeighborsClassifier()
    if model_name == 'MLP':
        return MLPClassifier(max_iter=3000, random_state=seed)
    if model_name == 'Logistic':
        return LogisticRegression(max_iter=3000, multi_class='auto', random_state=seed)
    raise ValueError(f'Unsupported model: {model_name}')


def make_pipeline(model_name: str, outlier_mode: str, seed: int):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if outlier_mode == 'winsorize_1_99':
        steps.append(('winsor', Winsorizer(0.01, 0.99)))
    elif outlier_mode != 'none':
        raise ValueError(f'Unsupported outlier mode: {outlier_mode}')

    if model_name in {'KNN', 'MLP', 'Logistic'}:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', get_model(model_name, seed)))
    return Pipeline(steps)


def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1_Macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Precision_Macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall_Macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }


## 3) Load and Prepare Train/External Data


In [6]:
train_df = load_tabular(CONFIG['input_path'], CONFIG['train_sheet'])
external_df = load_tabular(CONFIG['input_path'], CONFIG['external_sheet'])

train_clean = clean_dataframe(train_df, CONFIG['features'], CONFIG['train_target'], CONFIG['drop_missing'])
external_clean = clean_dataframe(external_df, CONFIG['features'], CONFIG['external_target'], CONFIG['drop_missing'])

X_all = train_clean[CONFIG['features']]
y_all_reg = train_clean[CONFIG['train_target']]
y_all_cls = bin_target(y_all_reg, CONFIG['bin_edges'], CONFIG['bin_labels'])

X_external = external_clean[CONFIG['features']]
y_external_reg = external_clean[CONFIG['external_target']]
y_external_cls = bin_target(y_external_reg, CONFIG['bin_edges'], CONFIG['bin_labels'])

print('Train shape:', train_clean.shape)
print('External shape:', external_clean.shape)
print('Train class counts:', y_all_cls.value_counts(dropna=False).sort_index())
print('External class counts:', y_external_cls.value_counts(dropna=False).sort_index())


Train shape: (273, 56)
External shape: (69, 94)
Train class counts: TREE
0-49       172
1000+        6
150-249     15
250-499      9
50-149      58
500-999     13
Name: count, dtype: int64
External class counts: REEY
0-49       43
150-249     6
250-499     8
50-149     11
500-999     1
Name: count, dtype: int64


## 4) Single-Seed Candidate Tuning (Internal Split)


In [7]:
idx = np.arange(len(X_all))
idx_train, idx_test = train_test_split(
    idx,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['single_seed'],
    stratify=y_all_cls,
)

X_train, X_test = X_all.iloc[idx_train], X_all.iloc[idx_test]
y_train, y_test = y_all_cls.iloc[idx_train], y_all_cls.iloc[idx_test]

single_seed_rows = []
cv = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['single_seed'])

for cand in CONFIG['candidates']:
    model_name = cand['name']
    outlier_mode = cand['outlier_mode']
    pipe = make_pipeline(model_name, outlier_mode, seed=CONFIG['single_seed'])

    if CONFIG['use_grid_search'] and model_name in PARAM_GRIDS:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=PARAM_GRIDS[model_name],
            scoring='f1_macro',
            cv=cv,
            n_jobs=CONFIG['grid_n_jobs'],
        )
        search.fit(X_train, y_train)
        best_estimator = search.best_estimator_
        best_params = search.best_params_
        cv_best = search.best_score_
    else:
        pipe.fit(X_train, y_train)
        best_estimator = pipe
        best_params = {}
        cv_best = np.nan

    y_pred = best_estimator.predict(X_test)
    m = classification_metrics(y_test, y_pred)

    single_seed_rows.append({
        'Model': model_name,
        'Outlier_Mode': outlier_mode,
        'CV_F1_Macro': cv_best,
        'Test_Accuracy': m['Accuracy'],
        'Test_F1_Macro': m['F1_Macro'],
        'Test_Precision_Macro': m['Precision_Macro'],
        'Test_Recall_Macro': m['Recall_Macro'],
        'Best_Params': best_params,
    })

single_seed_df = pd.DataFrame(single_seed_rows).sort_values('Test_F1_Macro', ascending=False).reset_index(drop=True)
single_seed_df


,Model,Outlier_Mode,CV_F1_Macro,Test_Accuracy,Test_F1_Macro,Test_Precision_Macro,Test_Recall_Macro,Best_Params
0,RandomForest,winsorize_1_99,0.632051,0.781818,0.586465,0.658333,0.550794,"{'model__max_depth': None, 'model__min_samples..."
1,MLP,winsorize_1_99,0.715457,0.818182,0.450794,0.458333,0.458333,"{'model__activation': 'relu', 'model__alpha': ..."
2,ExtraTrees,winsorize_1_99,0.622256,0.781818,0.404602,0.418026,0.398016,"{'model__max_depth': 20, 'model__min_samples_l..."
3,KNN,none,0.602775,0.781818,0.344949,0.345833,0.347222,"{'model__n_neighbors': 9, 'model__p': 2, 'mode..."


## 5) Multi-Seed Robustness (Internal Split)


In [9]:
def run_multiseed_internal(X_all, y_all_cls, candidates, seed_start, seed_count, test_size, cv_folds):
    rows = []
    idx = np.arange(len(X_all))
    for seed in range(seed_start, seed_start + seed_count):
        idx_train, idx_test = train_test_split(
            idx, test_size=test_size, random_state=seed, stratify=y_all_cls
        )
        X_train, X_test = X_all.iloc[idx_train], X_all.iloc[idx_test]
        y_train, y_test = y_all_cls.iloc[idx_train], y_all_cls.iloc[idx_test]

        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)

        for cand in candidates:
            model_name = cand['name']
            outlier_mode = cand['outlier_mode']
            pipe = make_pipeline(model_name, outlier_mode, seed=seed)

            if model_name in PARAM_GRIDS and CONFIG['use_grid_search']:
                search = GridSearchCV(
                    estimator=pipe,
                    param_grid=PARAM_GRIDS[model_name],
                    scoring='f1_macro',
                    cv=cv,
                    n_jobs=CONFIG['grid_n_jobs'],
                )
                search.fit(X_train, y_train)
                est = search.best_estimator_
                cv_f1 = search.best_score_
            else:
                est = pipe.fit(X_train, y_train)
                cv_f1 = np.nan

            pred = est.predict(X_test)
            m = classification_metrics(y_test, pred)
            rows.append({
                'Seed': seed,
                'Model': model_name,
                'Outlier_Mode': outlier_mode,
                'CV_F1_Macro': cv_f1,
                'Test_Accuracy': m['Accuracy'],
                'Test_F1_Macro': m['F1_Macro'],
                'Test_Precision_Macro': m['Precision_Macro'],
                'Test_Recall_Macro': m['Recall_Macro'],
            })

    per_seed = pd.DataFrame(rows)
    summary = per_seed.groupby(['Model', 'Outlier_Mode'], as_index=False).agg(
        Seeds=('Seed', 'count'),
        Test_Accuracy_Median=('Test_Accuracy', 'median'),
        Test_F1_Macro_Median=('Test_F1_Macro', 'median'),
        Test_F1_Macro_Mean=('Test_F1_Macro', 'mean'),
        Test_F1_Macro_Std=('Test_F1_Macro', 'std'),
    )
    q = per_seed.groupby(['Model', 'Outlier_Mode'])['Test_F1_Macro'].quantile([0.025, 0.975]).unstack().reset_index()
    q.columns = ['Model', 'Outlier_Mode', 'Test_F1_Macro_P2_5', 'Test_F1_Macro_P97_5']
    summary = summary.merge(q, on=['Model', 'Outlier_Mode'], how='left')
    summary = summary.sort_values(['Test_F1_Macro_P2_5', 'Test_F1_Macro_Median'], ascending=False).reset_index(drop=True)
    return per_seed, summary

internal_per_seed, internal_summary = run_multiseed_internal(
    X_all=X_all,
    y_all_cls=y_all_cls,
    candidates=CONFIG['candidates'],
    seed_start=CONFIG['seed_start'],
    seed_count=CONFIG['seed_count'],
    test_size=CONFIG['test_size'],
    cv_folds=CONFIG['cv_folds'],
)

internal_summary


/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/opt/anaconda3/envs/homl3/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by use

KeyboardInterrupt: 

## 6) External Validation (Train on Literature, Test on External)


In [ ]:
def run_multiseed_external(X_train, y_train_cls, X_external, y_external_cls, candidates, seed_start, seed_count, cv_folds):
    rows = []
    for seed in range(seed_start, seed_start + seed_count):
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)
        for cand in candidates:
            model_name = cand['name']
            outlier_mode = cand['outlier_mode']
            pipe = make_pipeline(model_name, outlier_mode, seed=seed)

            if model_name in PARAM_GRIDS and CONFIG['use_grid_search']:
                search = GridSearchCV(
                    estimator=pipe,
                    param_grid=PARAM_GRIDS[model_name],
                    scoring='f1_macro',
                    cv=cv,
                    n_jobs=CONFIG['grid_n_jobs'],
                )
                search.fit(X_train, y_train_cls)
                est = search.best_estimator_
                cv_f1 = search.best_score_
            else:
                est = pipe.fit(X_train, y_train_cls)
                cv_f1 = np.nan

            pred_ext = est.predict(X_external)
            m = classification_metrics(y_external_cls, pred_ext)
            rows.append({
                'Seed': seed,
                'Model': model_name,
                'Outlier_Mode': outlier_mode,
                'CV_F1_Macro': cv_f1,
                'External_Accuracy': m['Accuracy'],
                'External_F1_Macro': m['F1_Macro'],
                'External_Precision_Macro': m['Precision_Macro'],
                'External_Recall_Macro': m['Recall_Macro'],
            })

    per_seed = pd.DataFrame(rows)
    summary = per_seed.groupby(['Model', 'Outlier_Mode'], as_index=False).agg(
        Seeds=('Seed', 'count'),
        External_Accuracy_Median=('External_Accuracy', 'median'),
        External_F1_Macro_Median=('External_F1_Macro', 'median'),
        External_F1_Macro_Mean=('External_F1_Macro', 'mean'),
        External_F1_Macro_Std=('External_F1_Macro', 'std'),
    )
    q = per_seed.groupby(['Model', 'Outlier_Mode'])['External_F1_Macro'].quantile([0.025, 0.975]).unstack().reset_index()
    q.columns = ['Model', 'Outlier_Mode', 'External_F1_Macro_P2_5', 'External_F1_Macro_P97_5']
    summary = summary.merge(q, on=['Model', 'Outlier_Mode'], how='left')
    summary = summary.sort_values(['External_F1_Macro_P2_5', 'External_F1_Macro_Median'], ascending=False).reset_index(drop=True)
    return per_seed, summary

external_per_seed, external_summary = run_multiseed_external(
    X_train=X_all,
    y_train_cls=y_all_cls,
    X_external=X_external,
    y_external_cls=y_external_cls,
    candidates=CONFIG['candidates'],
    seed_start=CONFIG['seed_start'],
    seed_count=CONFIG['seed_count'],
    cv_folds=CONFIG['cv_folds'],
)

external_summary


## 7) Select Production Candidate


In [ ]:
selection_table = external_summary.copy().sort_values(
    ['External_F1_Macro_P2_5', 'External_F1_Macro_Median', 'External_Accuracy_Median'],
    ascending=False
).reset_index(drop=True)
selection_table.insert(0, 'Rank', np.arange(1, len(selection_table) + 1))
selection_table


In [ ]:
best = selection_table.iloc[0]
BEST_MODEL = best['Model']
BEST_OUTLIER = best['Outlier_Mode']
print('Selected production candidate:', {'Model': BEST_MODEL, 'Outlier_Mode': BEST_OUTLIER})


## 8) Fit Production Classifier and Evaluate on External


In [ ]:
production_pipe = make_pipeline(BEST_MODEL, BEST_OUTLIER, seed=CONFIG['single_seed'])

if CONFIG['use_grid_search'] and BEST_MODEL in PARAM_GRIDS:
    cv_full = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['single_seed'])
    final_search = GridSearchCV(
        estimator=production_pipe,
        param_grid=PARAM_GRIDS[BEST_MODEL],
        scoring='f1_macro',
        cv=cv_full,
        n_jobs=CONFIG['grid_n_jobs'],
    )
    final_search.fit(X_all, y_all_cls)
    production_model = final_search.best_estimator_
    print('Final best params:', final_search.best_params_)
else:
    production_model = production_pipe.fit(X_all, y_all_cls)

prod_pred_ext = production_model.predict(X_external)
prod_metrics = classification_metrics(y_external_cls, prod_pred_ext)
print('External metrics for production model:', prod_metrics)
print('
Classification report (external):')
print(classification_report(y_external_cls, prod_pred_ext, zero_division=0))


## 9) Visual Exports (Confusion Matrix + Class Distribution)


In [ ]:
VIS_DIR = Path('/Users/rowanterra/Desktop/DiGiTerra/docs/notebook_classification_visual_exports')
VIS_DIR.mkdir(parents=True, exist_ok=True)

def export_confusion_matrix(y_true, y_pred, labels, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(8, 6))
    if _HAS_SEABORN:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    else:
        im = ax.imshow(cm, cmap='Blues')
        fig.colorbar(im, ax=ax)
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_yticklabels(labels)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i, j], ha='center', va='center')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


export_confusion_matrix(
    y_true=y_external_cls,
    y_pred=prod_pred_ext,
    labels=CONFIG['bin_labels'],
    title=f'Production Confusion Matrix | {BEST_MODEL}_{BEST_OUTLIER}',
    out_path=VIS_DIR / 'production_external_confusion_matrix.png',
)

class_dist = pd.DataFrame({
    'Class': CONFIG['bin_labels'],
    'Train_Count': y_all_cls.value_counts().reindex(CONFIG['bin_labels']).fillna(0).astype(int).values,
    'External_Count': y_external_cls.value_counts().reindex(CONFIG['bin_labels']).fillna(0).astype(int).values,
})
class_dist.to_csv(VIS_DIR / 'class_distribution.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(CONFIG['bin_labels']))
w = 0.35
ax.bar(x - w/2, class_dist['Train_Count'], width=w, label='Train')
ax.bar(x + w/2, class_dist['External_Count'], width=w, label='External')
ax.set_xticks(x)
ax.set_xticklabels(CONFIG['bin_labels'], rotation=30, ha='right')
ax.set_ylabel('Count')
ax.set_title('Class Distribution by Bin')
ax.legend()
fig.tight_layout()
fig.savefig(VIS_DIR / 'class_distribution.png', dpi=180)
plt.close(fig)

print('Saved visuals to', VIS_DIR)


## 10) Predict on a New Dataset


In [ ]:
NEW_DATA_PATH = Path('/absolute/path/to/new_dataset.xlsx')
NEW_DATA_SHEET = 'Sheet1'

# Uncomment to run:
# new_df = load_tabular(NEW_DATA_PATH, NEW_DATA_SHEET if NEW_DATA_PATH.suffix.lower() in {'.xlsx', '.xls'} else None)
# missing_new = [c for c in CONFIG['features'] if c not in new_df.columns]
# if missing_new:
#     raise ValueError(f'Missing required feature columns in new data: {missing_new}')
#
# X_new = new_df[CONFIG['features']].apply(pd.to_numeric, errors='coerce')
# pred_class = production_model.predict(X_new)
#
# pred_df = new_df.copy()
# pred_df['Predicted_Class'] = pred_class
#
# if hasattr(production_model, 'predict_proba'):
#     proba = production_model.predict_proba(X_new)
#     for i, cls in enumerate(production_model.classes_):
#         pred_df[f'Prob_{cls}'] = proba[:, i]
#
# out_path = Path('/Users/rowanterra/Desktop/DiGiTerra/docs/new_data_class_predictions.csv')
# pred_df.to_csv(out_path, index=False)
# print('Saved predictions to', out_path)


## 11) Save Tables


In [5]:
OUT_DIR = Path('/Users/rowanterra/Desktop/DiGiTerra/docs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

single_seed_df.to_csv(OUT_DIR / 'notebook_cls_single_seed_results.csv', index=False)
internal_per_seed.to_csv(OUT_DIR / 'notebook_cls_internal_per_seed.csv', index=False)
internal_summary.to_csv(OUT_DIR / 'notebook_cls_internal_summary.csv', index=False)
external_per_seed.to_csv(OUT_DIR / 'notebook_cls_external_per_seed.csv', index=False)
external_summary.to_csv(OUT_DIR / 'notebook_cls_external_summary.csv', index=False)
selection_table.to_csv(OUT_DIR / 'notebook_cls_selection_table.csv', index=False)

print('Saved classification notebook outputs to', OUT_DIR)


NameError: name 'single_seed_df' is not defined